# Wine Quality (Binary) — Full Experiment Notebook

HyperAck-style walkthrough for **Wine Quality (Binary)**.

- Source: https://archive.ics.uci.edu/dataset/186/wine+quality
- Leakage columns (unsafe-only): `[]`
- Has leakage: **False**
- Policy: Physicochemical tests are available before quality rating; no leakage identified.

This notebook mirrors the HyperAck ladder: baseline → FE families → selection → tuning → calibration → ensembles → safe vs unsafe honesty check.


## 0. Setup

In [ ]:

from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name.endswith('_exp'):
    ROOT = ROOT.parents[1]
elif (ROOT / 'general_pipeline').exists() is False:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'hyperack_exp'))

from general_pipeline.playbook.ladder import run_project_ladder, results_frame, load_raw_xy
from general_pipeline.playbook.features import build_feature_matrix
from general_pipeline.playbook.reports import write_detailed_reports
from general_pipeline.playbook.policy import get_policy

KEY = 'wine_quality'
policy = get_policy(KEY)
print(policy)


## 1. Load safe vs unsafe splits
Safe drops post-outcome / contested columns when defined.

In [ ]:

Xtr_s, ytr_s, Xte_s, yte_s, meta_s = load_raw_xy(KEY, 'safe')
Xtr_u, ytr_u, Xte_u, yte_u, meta_u = load_raw_xy(KEY, 'unsafe')
print('Safe features:', Xtr_s.shape, 'dropped:', meta_s.get('dropped_leakage'))
print('Unsafe features:', Xtr_u.shape, 'dropped:', meta_u.get('dropped_leakage'))
print('Pos rate:', meta_s.get('pos_rate', ytr_s.mean()))


## 2. Feature engineering stages
Inspect how each FE stage expands the matrix (train-fit stateful steps).

In [ ]:

for stage in ['raw', 'logs', 'ratios', 'interactions', 'full_fe', 'selected']:
    Xtr, Xte, fe_meta = build_feature_matrix(Xtr_s, ytr_s, Xte_s, stage=stage)
    print(f'{stage:12s} -> train {Xtr.shape} | top MI: {fe_meta.get("top_mi", [])[:5]}')


## 3. Run full ladder (safe + unsafe)
15 experiments × modes — same protocol as HyperAck 01–15.

In [ ]:

# Uncomment to re-run (can take several minutes):
# df = run_project_ladder(KEY)
df = results_frame(KEY)
df.sort_values(['mode', 'exp_id']).head(20)


## 4. Safe vs Unsafe leakage cost

In [ ]:

import pandas as pd
safe = df[df['mode']=='safe'].set_index('exp_name')['roc_auc']
unsafe = df[df['mode']=='unsafe'].set_index('exp_name')['roc_auc']
cmp = pd.DataFrame({'safe': safe, 'unsafe': unsafe}).dropna()
cmp['leakage_gap'] = cmp['unsafe'] - cmp['safe']
cmp.sort_values('leakage_gap', ascending=False)


## 5. Which optimization method wins?

In [ ]:

opt = df[df['mode']=='safe'][['exp_name','optimization_method','roc_auc','f1','feature_count']].sort_values('roc_auc', ascending=False)
opt


## 6. Regenerate detailed reports + figures

In [ ]:

write_detailed_reports(KEY, df)
print('Reports written to external_projects/%s_exp/' % KEY)


## 7. Next steps for a new dataset

1. Add entry to `general_pipeline/external_catalog.py` and download.
2. Define leakage in `general_pipeline/playbook/policy.py`.
3. Run `python general_pipeline/playbook/run_playbook.py --dataset <key>`.
4. Open the generated notebook and reports in `external_projects/<key>_exp/`.
